In [ ]:
import math
import json
import numpy as np
from utils import get_topics

In [ ]:
data_path = 'path_to_ap_news_data'
raw_corpus_features_path = 'path_to_raw_corpus_features/'
corpus_features_path = 'path_to_corpus_features/'
lda_logs = 'path_to_lda_logs/'
ntm_logs_root = 'path_to_root_ntm_log_dir/'
wete_logs = 'path_to_wete_logs/'
fastopic_logs = 'path_to_fastopic_logs/'

In [3]:
def bow(doc):
    words = doc.split(' ')[1:]
    words = [w.split(':') for w in words]
    words = {vocab[int(w[0])]: int(w[1].strip('\n')) for w in words}
    return words

K = 10
data = []
with open(f'{raw_corpus_features_path}/train.feat', 'r') as f:
    data += f.readlines()
with open(f'{raw_corpus_features_path}/test.feat', 'r') as f:
    data += f.readlines()
with open(f'{raw_corpus_features_path}/vocab', 'r') as f:
    vocab = f.readlines()
vocab = [w.split(' ') for w in vocab]
vocab = {int(w[1].strip('\n')): w[0] for w in vocab}
bow_data = [bow(doc) for doc in data]
D = len(bow_data)

V = len(vocab)
occurence_sets = {vocab[i]: set({}) for i in range(1,V+1)}
for i, doc in enumerate(bow_data):
    for k in doc.keys():
        occurence_sets[k].add(i)

def occurences(w, p=False):
    c = len(occurence_sets[w])
    return c / D if p else c

def co_occurences(w1, w2, p=False):
    c1 = occurence_sets[w1]
    c2 = occurence_sets[w2]
    c = len(c1.intersection(c2))
    return c / D if p else c

In [4]:
def umass(topic):
    score = 0
    k = len(topic)
    if k <= 1:
        return None
    for i in range(k):
        for j in range(i+1,k):
            w1 = topic[i]
            w2 = topic[j]
            p1 = occurences(w2, p=True)
            p12 = co_occurences(w1, w2, p=True)
            if p12 < p1:
                score += math.log((p12 + 1/D) / p1, 10)
    score /= k * (k-1) // 2
    return score

def lcp(topic):
    score = 0
    k = len(topic)
    if k <= 1:
        return None
    for i in range(k):
        for j in range(i+1,k):
            w1 = topic[i]
            w2 = topic[j]
            p1 = occurences(w1, p=True)
            p2 = occurences(w2, p=True)
            p12 = co_occurences(w1, w2, p=True)
            if p12 > 0:
                score += math.log(p12 / p1, 10)
            else:
                score += math.log(p2, 10)
    score /= k * (k-1) // 2
    return score

def pmi(topic):
    score = 0
    k = len(topic)
    if k <= 1:
        return None
    for i in range(k):
        for j in range(i+1,k):
            w1 = topic[i]
            w2 = topic[j]
            p1 = occurences(w1, p=True)
            p2 = occurences(w2, p=True)
            p12 = co_occurences(w1, w2, p=True)
            if p12 > 0:
                score += math.log(p12 / (p1 * p2), 10)
    score /= k * (k-1) // 2
    return score

def npmi(topic):
    score = 0
    k = len(topic)
    if k <= 1:
        return None
    for i in range(k):
        for j in range(i+1,k):
            w1 = topic[i]
            w2 = topic[j]
            p1 = occurences(w1, p=True)
            p2 = occurences(w2, p=True)
            p12 = co_occurences(w1, w2, p=True)
            if p12 > 0:
                pmi = math.log(p12 / (p1 * p2), 10)
                score += -pmi / math.log(p12, 10)
    score /= k * (k-1) // 2
    return score

def topic_coherence(topics, func):
    scores = [func(topic) for topic in topics]
    scores = [s for s in scores if s is not None]
    score = sum(scores) / len(scores)
    return (score, max(scores))

def update(arr, npmi_scores, umass_scores, seed):
    arr[0][seed-42] = npmi_scores[0]
    arr[1][seed-42] = npmi_scores[1]
    arr[2][seed-42] = umass_scores[0]
    arr[3][seed-42] = umass_scores[1]


In [5]:
mpm = np.zeros((4,5))
for seed in range(42,47):
    mpm_topics = get_topics(seed) 
    npmi_scores = topic_coherence(mpm_topics, npmi)
    umass_scores = topic_coherence(mpm_topics, umass)
    update(mpm, npmi_scores, umass_scores, seed)

In [6]:
data = []
with open(f'{corpus_features_path}/train.feat', 'r') as f:
    data += f.readlines()
with open(f'{corpus_features_path}/test.feat', 'r') as f:
    data += f.readlines()
with open(f'{corpus_features_path}/vocab', 'r') as f:
    vocab = f.readlines()
vocab = [w.split(' ') for w in vocab]
vocab = {int(w[1].strip('\n')): w[0] for w in vocab}
bow_data = [bow(doc) for doc in data]
D = len(bow_data)

K = 10
V = len(vocab)
occurence_sets = {vocab[i]: set({}) for i in range(1,V+1)}
for i, doc in enumerate(bow_data):
    for k in doc.keys():
        occurence_sets[k].add(i)

In [7]:
lda = np.zeros((4,5))
nvdm = np.zeros((4,5))
gsm = np.zeros((4,5))
ntm = np.zeros((4,5))
ntmr = np.zeros((4,5))
wete = np.zeros((4,5))
fast = np.zeros((4,5))

for seed in range(42,47):
    with open(f'{lda_logs}/topic-100-seed={seed}.txt', 'r') as f:
        lda_topics = f.readlines()
    lda_topics = [topic.strip('\n') for topic in lda_topics]
    lda_topics = [topic.split(' ')[:K] for topic in lda_topics]
    npmi_scores = topic_coherence(lda_topics, npmi)
    umass_scores = topic_coherence(lda_topics, umass)
    update(lda, npmi_scores, umass_scores, seed)

    arrs = [gsm, ntm, ntmr, nvdm]
    names = ['gsm', 'ntm', 'ntmr', 'nvdm']
    for name, arr in zip(names, arrs):
        ntm_dir = f'{ntm_logs_root}/{name}/{seed-41}'
        with open(f'{ntm_dir}/best_eval.json', 'r') as f:
            data = json.load(f)
        epoch = data['epoch']
        with open(f'{ntm_dir}/topic-{epoch}.topics', 'r') as f:
            ntm_topics = f.readlines()
        ntm_topics = [topic.split(' ')[:K] for topic in ntm_topics]
        npmi_scores = topic_coherence(ntm_topics, npmi)
        umass_scores = topic_coherence(ntm_topics, umass)
        update(arr, npmi_scores, umass_scores, seed)

    wete_dir = f'{wete_logs}/runs/ap/k_100/seed={seed}'
    with open(f'{wete_dir}/phi_490.txt', 'r') as f:
        wete_topics = f.readlines()
    wete_topics = [topic.split(' ')[:K] for topic in wete_topics]
    npmi_scores = topic_coherence(wete_topics, npmi)
    umass_scores = topic_coherence(wete_topics, umass)
    update(wete, npmi_scores, umass_scores, seed)
    
    with open(f'{fastopic_logs}/topic-100-seed={seed}.txt', 'r') as f:
        fast_topics = f.readlines()
    fast_topics = [topic.strip('\n') for topic in fast_topics]
    fast_topics = [topic.split(' ')[:K] for topic in fast_topics]
    npmi_scores = topic_coherence(fast_topics, npmi)
    umass_scores = topic_coherence(fast_topics, umass)
    update(fast, npmi_scores, umass_scores, seed)

In [8]:
latex = False
def pprint(scores):
    s = []
    if latex:
        for i in range(4):
            s.append(f'${scores[i].mean():.03f} \pm {scores[i].std():.03f}$')
        return ' & '.join(s)
    else:
        for i in range(4):
            s.append(f'{scores[i].mean():.03f} ± {scores[i].std():.03f}')
        return ' | '.join(s)

npmi_header = '-'*11 + ' NPMI ' + '-'*12
umass_header = '-'*12 + ' UMASS ' + '-'*12
print('     ', npmi_header, ' ', umass_header)
print('LDA :', pprint(lda))
print('NVDM:', pprint(nvdm))
print('GSM :', pprint(gsm))
print('NTM :', pprint(ntm))
print('NTMR:', pprint(ntmr))
print('WeTe:', pprint(wete))
print('FAST:', pprint(fast))
print('MPM :', pprint(mpm))

      ----------- NPMI ------------   ------------ UMASS ------------
LDA : 0.165 ± 0.005 | 0.599 ± 0.145 | -0.753 ± 0.015 | -0.215 ± 0.076
NVDM: 0.070 ± 0.002 | 0.157 ± 0.011 | -1.072 ± 0.028 | -0.746 ± 0.065
GSM : 0.145 ± 0.003 | 0.424 ± 0.041 | -0.500 ± 0.049 | -0.224 ± 0.026
NTM : 0.217 ± 0.011 | 0.465 ± 0.045 | -0.700 ± 0.030 | -0.418 ± 0.026
NTMR: 0.072 ± 0.001 | 0.143 ± 0.016 | -1.071 ± 0.011 | -0.704 ± 0.030
WeTe: 0.230 ± 0.003 | 0.617 ± 0.028 | -0.589 ± 0.015 | -0.167 ± 0.115
FAST: 0.207 ± 0.004 | 0.724 ± 0.067 | -0.462 ± 0.019 | -0.012 ± 0.012
MPM : 0.138 ± 0.022 | 0.624 ± 0.181 | -0.499 ± 0.059 | -0.011 ± 0.008
